In [1]:
# ==== IMPORTS ====
from pathlib import Path
import re
import fitz
import pandas as pd
import spacy

In [2]:
import sys
import subprocess

# Ensure pip exists in this exact kernel environment
subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])

# Download spaCy model into this environment
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
print("spaCy model installed: en_core_web_sm")

Looking in links: /var/folders/kb/z559lc5s7jzbc1c4j024j2w40000gn/T/tmpn0g688ed
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
spaCy model installed: en_core_web_sm


In [3]:
def extract_linguistics_metrics(pdf_path: str, debug: bool = False) -> dict | None:
    """Extract linguistics metrics from thesis PDF, scoped to main content only."""
    try:
        nlp = spacy.load("en_core_web_sm")

        doc = fitz.open(pdf_path)
        num_tot_pages = len(doc)
        min_end_page = max(1, int(num_tot_pages * 0.30))

        end_boundary_exact = {
            "references",
            "bibliography",
            "works cited",
            "list of references",
            "reference list",
            "appendix",
            "appendices",
            "referencer",
            "bibliografi",
            "litteratur",
            "litteraturliste",
            "litteraturfortegnelse",
            "kildeliste",
            "bilag",
            "appendiks",
            "list of figures",
            "list of tables",
        }
        end_boundary_prefix = (
            "references",
            "bibliography",
            "works cited",
            "appendix",
            "appendices",
            "referencer",
            "bibliografi",
            "litteratur",
            "kildeliste",
            "bilag",
            "appendiks",
        )

        def _is_toc_context(lines: list[str], heading_line_num: int) -> bool:
            """Reject heading detections that are likely part of a TOC page/context."""
            pre_lines = [ln.strip() for ln in lines[:heading_line_num] if ln.strip()]
            context = " ".join(pre_lines).lower()

            toc_markers = (
                "contents",
                "table of contents",
                "indholdsfortegnelse",
                "preface",
                "acknowledgements",
            )
            if any(marker in context for marker in toc_markers):
                return True

            dot_leader_pattern = re.compile(r"(?:\.\s*){4,}\d{1,3}\s*$")
            trailing_page_no_pattern = re.compile(r"\b\d{1,3}\s*$")
            numeric_only_pattern = re.compile(r"^\d{1,3}$")
            toc_tail_markers = ("figurer", "figures", "tabeller", "tables", "bilag", "appendix")

            toc_like_lines = 0
            for ln in pre_lines:
                if dot_leader_pattern.search(ln):
                    toc_like_lines += 1
                    continue
                if trailing_page_no_pattern.search(ln) and re.search(r"[A-Za-zÆØÅæøå]", ln):
                    toc_like_lines += 1

            post_lines = [ln.strip() for ln in lines[heading_line_num + 1 : heading_line_num + 12] if ln.strip()]
            toc_like_post = 0
            post_numeric_only = 0
            post_toc_marker_hits = 0
            for ln in post_lines:
                if dot_leader_pattern.search(ln):
                    toc_like_post += 1
                    continue
                if trailing_page_no_pattern.search(ln) and re.search(r"[A-Za-zÆØÅæøå]", ln):
                    toc_like_post += 1
                if numeric_only_pattern.match(ln):
                    post_numeric_only += 1
                if any(marker in ln.lower() for marker in toc_tail_markers):
                    post_toc_marker_hits += 1

            if toc_like_post >= 3:
                return True
            if post_numeric_only >= 3 and post_toc_marker_hits >= 2:
                return True

            return toc_like_lines >= 6

        def _find_main_content_end_page() -> tuple[int, str | None]:
            """Reuse local_metrics boundary logic: return number of content pages and trigger."""
            num_cont_pages = 0
            match_trigger = None

            for page_number, page in enumerate(doc, start=1):
                text = page.get_text("text") or ""
                lines = [line.strip().lower() for line in text.splitlines() if line.strip()]

                matched_line = None
                local_trigger = None

                for line_idx, line in enumerate(lines):
                    tokens = line.split()
                    prefix_token = None
                    core_line = line

                    if tokens:
                        first_token = tokens[0].rstrip(").:-")
                        if first_token.isdigit() or (
                            len(first_token) == 1 and first_token.isalpha()
                        ):
                            prefix_token = first_token
                            core_line = " ".join(tokens[1:]).strip()

                    if core_line and core_line in end_boundary_exact:
                        if prefix_token and prefix_token.isdigit():
                            local_trigger = (
                                f"numeric-prefix exact ('{prefix_token} {core_line}')"
                            )
                        elif prefix_token:
                            local_trigger = (
                                f"letter-prefix exact ('{prefix_token} {core_line}')"
                            )
                        else:
                            local_trigger = f"exact ('{core_line}')"
                    elif core_line and any(core_line.startswith(p) for p in end_boundary_prefix):
                        matched_prefix = next(
                            p for p in end_boundary_prefix if core_line.startswith(p)
                        )
                        if prefix_token and prefix_token.isdigit():
                            local_trigger = (
                                f"numeric-prefix prefix-match ('{prefix_token} {core_line}', prefix='{matched_prefix}')"
                            )
                        elif prefix_token:
                            local_trigger = (
                                f"letter-prefix prefix-match ('{prefix_token} {core_line}', prefix='{matched_prefix}')"
                            )
                        else:
                            local_trigger = (
                                f"prefix-match ('{core_line}', prefix='{matched_prefix}')"
                            )
                    else:
                        local_trigger = None

                    if local_trigger is None:
                        continue

                    words = line.split()
                    has_short_length = len(line) <= 60 and len(words) <= 8
                    ends_with_punctuation = line.endswith(",") or line.endswith(";") or line.endswith(":") or line.endswith(".") or line.endswith(")")

                    core_words = core_line.split()
                    first_core_token = core_words[0] if core_words else ""
                    trailing_words = (
                        core_words[1:] if first_core_token in end_boundary_exact else core_words
                    )
                    lowercase_trailing_count = sum(
                        1 for w in trailing_words if w.isalpha() and w.islower()
                    )
                    has_many_lowercase_trailing = lowercase_trailing_count >= 4

                    if not has_short_length:
                        continue
                    if ends_with_punctuation:
                        continue
                    if has_many_lowercase_trailing:
                        continue

                    # New hardening: ignore TOC-context matches.
                    if _is_toc_context(lines, line_idx):
                        continue

                    matched_line = line
                    break

                if matched_line is not None:
                    if page_number > min_end_page:
                        match_trigger = local_trigger
                        break

                num_cont_pages += 1

            return num_cont_pages, match_trigger

        num_cont_pages, match_trigger = _find_main_content_end_page()

        main_text_parts = []
        for page in doc[:num_cont_pages]:
            main_text_parts.append(page.get_text("text") or "")
        doc.close()

        main_text = "\n".join(main_text_parts)
        main_text = re.sub(r"\s+", " ", main_text).strip()
        if not main_text:
            return None

        nlp_doc = nlp(main_text[:200000])

        sentences = list(nlp_doc.sents)
        words = [token.text for token in nlp_doc if not token.is_punct and not token.is_space]

        if not sentences or not words:
            return None

        total_sentences = len(sentences)
        total_words = len(words)
        unique_words = len(set(w.lower() for w in words))

        avg_sentence_length = total_words / total_sentences
        avg_word_length = sum(len(w) for w in words) / total_words if total_words > 0 else 0
        lexical_diversity = unique_words / total_words if total_words > 0 else 0

        syllable_count = sum(count_syllables(w) for w in words)
        fk_grade = (0.39 * total_words / total_sentences) + (11.8 * syllable_count / total_words) - 15.59
        fk_grade = max(0, min(18, fk_grade))

        metrics = {
            "total_sentences": total_sentences,
            "total_words": total_words,
            "unique_words": unique_words,
            "avg_sentence_length": round(avg_sentence_length, 2),
            "avg_word_length": round(avg_word_length, 2),
            "lexical_diversity": round(lexical_diversity, 3),
            "flesch_kincaid_grade": round(fk_grade, 1),
        }

        if debug:
            print(f"[Linguistics] {Path(pdf_path).name}")
            print(f"  num_tot_pages: {num_tot_pages}")
            print(f"  num_cont_pages: {num_cont_pages}")
            print(f"  content_end_trigger: {match_trigger}")
            for k in (
                "total_sentences",
                "total_words",
                "unique_words",
                "avg_sentence_length",
                "avg_word_length",
                "lexical_diversity",
                "flesch_kincaid_grade",
            ):
                print(f"  {k}: {metrics[k]}")

        return metrics

    except Exception as e:
        print(f"[extract_linguistics_metrics] Failed for {pdf_path}: {e}")
        return None


def count_syllables(word: str) -> int:
    """Approximate syllable count for Flesch-Kincaid calculation."""
    word = word.lower()
    vowels = "aeiou"
    syllable_count = 0
    previous_was_vowel = False

    for char in word:
        is_vowel = char in vowels
        if is_vowel and not previous_was_vowel:
            syllable_count += 1
        previous_was_vowel = is_vowel

    if word.endswith("e"):
        syllable_count -= 1

    return max(1, syllable_count)

In [4]:
# Flexible linguistics run: choose how many files to process

local_pdf_root = Path("../Data/RAW_test")
pdf_paths = sorted(local_pdf_root.rglob("*.pdf"))

# Build candidate file list (prefer validation list when available)
if "validation" in locals() and "pdf_file" in validation.columns:
    candidate_files = [str(x).strip() for x in validation["pdf_file"].dropna().tolist()]
else:
    candidate_files = sorted([p.name for p in local_pdf_root.glob("*.pdf")])

if not candidate_files:
    print(f"No PDF files found in: {local_pdf_root}")
else:
    total_files = len(candidate_files)
    print(f"Found {total_files} PDF file(s) for linguistics in: {local_pdf_root}")

    user_input = input(f"How many files do you want to process? (1-{total_files}): ").strip()

    try:
        num_to_process = int(user_input)
        if num_to_process <= 0:
            raise ValueError
    except ValueError:
        raise SystemExit("Error: Invalid input. Please enter a positive integer. Execution terminated.")

    if num_to_process > total_files:
        choice = input(
            f"Error: Requested {num_to_process} files, but only {total_files} available.\n"
            "Type 'all' to process all files, or 'q' to terminate: "
        ).strip().lower()

        if choice == "all":
            num_to_process = total_files
        elif choice in {"q", "quit"}:
            raise SystemExit("Execution terminated by user. No files were processed.")
        else:
            raise SystemExit("Error: Invalid choice. Execution terminated without processing files.")

    selected_files = candidate_files[:num_to_process]

    linguistics_results = []
    failed_files = []

    for current_file_number, file_name in enumerate(selected_files, start=1):
        pdf_path = local_pdf_root / file_name
        print(f"\n=== Processing file {current_file_number} of {num_to_process} ===")
        print(f"Current file is: {file_name}")

        if not pdf_path.exists():
            print("File not found, skipping.")
            failed_files.append(file_name)
            continue

        result = extract_linguistics_metrics(str(pdf_path), debug=False)
        if result is None:
            failed_files.append(file_name)
            continue

        linguistics_results.append({"pdf_file": file_name, **result})

    linguistics_df = pd.DataFrame(linguistics_results)

    print("\n" + "=" * 100)
    print("LINGUISTICS METRICS SUMMARY (Main Content Only)")
    print("=" * 100)
    display(linguistics_df)

    if not linguistics_df.empty:
        print("\nSTATISTICS ACROSS SELECTED FILES:")
        print(f"  Avg Sentence Length: {linguistics_df['avg_sentence_length'].mean():.2f} ± {linguistics_df['avg_sentence_length'].std():.2f}")
        print(f"  Avg Word Length: {linguistics_df['avg_word_length'].mean():.2f} ± {linguistics_df['avg_word_length'].std():.2f}")
        print(f"  Lexical Diversity: {linguistics_df['lexical_diversity'].mean():.3f} ± {linguistics_df['lexical_diversity'].std():.3f}")
        print(f"  Flesch-Kincaid Grade: {linguistics_df['flesch_kincaid_grade'].mean():.1f} ± {linguistics_df['flesch_kincaid_grade'].std():.1f}")

    if failed_files:
        print(f"\n⚠ Failed files ({len(failed_files)}):")
        for ff in failed_files:
            print(f"  - {ff}")

    print(f"\n✓ Linguistics processing complete! ({len(linguistics_df)} succeeded / {num_to_process} requested)")

Found 50 PDF file(s) for linguistics in: ../Data/RAW_test

=== Processing file 1 of 2 ===
Current file is: 5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf

=== Processing file 2 of 2 ===
Current file is: 5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf

LINGUISTICS METRICS SUMMARY (Main Content Only)


,pdf_file,total_sentences,total_words,unique_words,avg_sentence_length,avg_word_length,lexical_diversity,flesch_kincaid_grade
0,5cee68eed9001d2064299318_Deep Reinforcement Le...,853,16784,2565,19.68,4.7,0.153,10.7
1,5cf3aee6d9001d2064299346_Human response to inc...,779,12908,2538,16.57,4.8,0.197,9.8



STATISTICS ACROSS SELECTED FILES:
  Avg Sentence Length: 18.12 ± 2.20
  Avg Word Length: 4.75 ± 0.07
  Lexical Diversity: 0.175 ± 0.031
  Flesch-Kincaid Grade: 10.2 ± 0.6

✓ Linguistics processing complete! (2 succeeded / 2 requested)
